# Notebook for testing out various models

### Models to Test : 
* ARIMA
* SARIMAX
* Prophet (formerly FBProphet)
* Random Forest & XGBoost
* LSTM
* GRU
* Transformer Models - BERT

In [1]:
# !pip install xgboost
# !pip install prophet
# !pip install torch
# !pip install lightgbm

## Libraries

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from prophet import Prophet
import torch

## Data

In [23]:
data = pd.read_csv("../data/MSFT.csv")
data.describe()

,Open,High,Low,Close,Volume,Day,DayOfWeek,Quarter,Month,Year,...,Rolling_Min,Rolling_Max,EMA_12,EMA_26,MACD,MACD_Signal,RSI,Overall_Mean,Overall_Min,Overall_Max
count,4628.000000,4628.000000,4628.000000,4628.000000,4.628000e+03,4628.000000,4628.000000,4628.000000,4628.000000,4628.000000,...,4628.000000,4628.000000,4628.000000,4628.000000,4628.0,4628.0,4628.000000,4628.000000,4628.000000,4628.000000
mean,97.955189,98.910500,96.971064,98.081905,4.368892e+07,15.724935,2.023984,2.493518,6.485523,2014.767070,...,96.878388,98.908216,97.496620,97.496620,0.0,0.0,54.814277,97.775456,11.283761,448.369995
std,108.974136,110.004939,107.888864,109.128267,2.826966e+07,8.755582,1.399028,1.113681,3.419077,5.315157,...,107.777817,110.011578,108.366746,108.366746,0.0,0.0,15.821376,0.000000,0.000000,0.000000
min,11.321001,11.633818,11.075216,11.283761,7.425600e+06,1.000000,0.000000,1.000000,1.000000,2006.000000,...,11.283761,11.380585,12.070885,12.070885,0.0,0.0,6.806019,97.775456,11.283761,448.369995
25%,21.497524,21.656163,21.284262,21.509594,2.491118e+07,8.000000,1.000000,2.000000,4.000000,2010.000000,...,21.303935,21.642164,21.474782,21.474782,0.0,0.0,43.544649,97.775456,11.283761,448.369995
50%,39.977358,40.305183,39.685150,40.030407,3.599870e+07,16.000000,2.000000,2.000000,6.000000,2015.000000,...,39.652710,40.360760,39.916460,39.916460,0.0,0.0,54.971384,97.775456,11.283761,448.369995
75%,138.313646,139.715566,136.777445,138.236942,5.406520e+07,23.000000,3.000000,3.000000,9.000000,2019.000000,...,134.335438,138.622772,137.553714,137.553714,0.0,0.0,66.380397,97.775456,11.283761,448.369995
max,442.589996,450.940002,440.720001,448.369995,5.910522e+08,31.000000,4.000000,4.000000,12.000000,2024.000000,...,441.579987,448.369995,433.735117,433.735117,0.0,0.0,99.111514,97.775456,11.283761,448.369995


In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4628 entries, 0 to 4627
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          4628 non-null   object 
 1   Open          4628 non-null   float64
 2   High          4628 non-null   float64
 3   Low           4628 non-null   float64
 4   Close         4628 non-null   float64
 5   Volume        4628 non-null   int64  
 6   Day           4628 non-null   int64  
 7   DayOfWeek     4628 non-null   int64  
 8   Quarter       4628 non-null   int64  
 9   Month         4628 non-null   int64  
 10  Year          4628 non-null   int64  
 11  S_3           4628 non-null   float64
 12  S_9           4628 non-null   float64
 13  S_18          4628 non-null   float64
 14  lag_1         4628 non-null   float64
 15  lag_2         4628 non-null   float64
 16  lag_3         4628 non-null   float64
 17  Rolling_Mean  4628 non-null   float64
 18  Rolling_Min   4628 non-null 

In [25]:
data

,Date,Open,High,Low,Close,Volume,Day,DayOfWeek,Quarter,Month,...,Rolling_Min,Rolling_Max,EMA_12,EMA_26,MACD,MACD_Signal,RSI,Overall_Mean,Overall_Min,Overall_Max
0,2006-01-27 05:00:00+00:00,19.234521,19.743110,19.206267,19.778437,134520700,27,4,1,1,...,18.648235,19.630091,18.951475,18.951475,0.0,0.0,62.643770,97.775456,11.283761,448.369995
1,2006-01-30 05:00:00+00:00,19.651289,19.905584,19.623035,19.884378,103999200,30,0,1,1,...,18.718876,19.778437,19.078700,19.078700,0.0,0.0,65.659794,97.775456,11.283761,448.369995
2,2006-01-31 05:00:00+00:00,19.714849,20.046844,19.686595,19.806681,94841300,31,1,1,1,...,19.630091,19.884378,19.202651,19.202651,0.0,0.0,65.753562,97.775456,11.283761,448.369995
3,2006-02-01 05:00:00+00:00,19.750170,19.827871,19.608896,19.552387,68448800,1,2,1,2,...,19.778437,19.884378,19.295578,19.295578,0.0,0.0,60.807027,97.775456,11.283761,448.369995
4,2006-02-02 05:00:00+00:00,19.757235,19.771362,19.460558,19.453501,55073400,2,3,1,2,...,19.552387,19.884378,19.335087,19.335087,0.0,0.0,57.337032,97.775456,11.283761,448.369995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4623,2024-06-11 04:00:00+00:00,425.480011,432.820007,425.250000,441.059998,14551100,11,1,2,6,...,423.850006,432.679993,424.081913,424.081913,0.0,0.0,53.910602,97.775456,11.283761,448.369995
4624,2024-06-12 04:00:00+00:00,435.320007,443.399994,433.250000,441.579987,22366200,12,2,2,6,...,427.869995,441.059998,426.693926,426.693926,0.0,0.0,59.861540,97.775456,11.283761,448.369995
4625,2024-06-13 04:00:00+00:00,440.850006,443.390015,439.369995,442.570007,15960600,13,3,2,6,...,432.679993,441.579987,428.984089,428.984089,0.0,0.0,64.452810,97.775456,11.283761,448.369995
4626,2024-06-14 04:00:00+00:00,438.279999,443.140015,436.720001,448.369995,13582000,14,4,2,6,...,441.059998,442.570007,431.074231,431.074231,0.0,0.0,62.854782,97.775456,11.283761,448.369995


## Data Preparation For ARIMA

In [65]:
# arima_data = data["Close"]
# arima_data

In [66]:
# train_arima = arima_data.iloc[:-2000]
# print(train_arima)
# test_arima = arima_data.iloc[-2000:]
# print(test_arima)

### ARIMA

In [67]:
# model = ARIMA(train_arima, order=(1,1,50))
# model_fit = model.fit()
# print(model_fit.summary())

In [68]:
# forecast_preds = model_fit.forecast(2000)
# print(forecast_preds)

In [69]:
# # Evaluation : 
# rmse = np.sqrt(mean_squared_error(forecast_preds,test_arima))
# print("Root Mean Squared Error : ",rmse) 
# mae = mean_absolute_error(forecast_preds,test_arima)
# print("Mean Absolute Error : ",mae) 
# # Model's performance is real bad, worse

In [70]:
# # Convert RangeIndex objects to Series
# test_data_dates = pd.Series(test_data.index)
# forecast_dates = pd.date_range(start=test_data.index[-1], periods=90, freq='D')
# forecast_dates = pd.Series(forecast_dates)

# forecast_dates = pd.concat([test_data_dates, forecast_dates])

# plt.plot(test_data["Close"], label='Actual')
# plt.plot(forecast, label='Forecast')

# plt.legend()
# plt.show()

### Random Forest Regressor with XGBoost 

In [71]:
data

,Date,Open,High,Low,Close,Volume,Day,DayOfWeek,Quarter,Month,...,Rolling_Min,Rolling_Max,EMA_12,EMA_26,MACD,MACD_Signal,RSI,Overall_Mean,Overall_Min,Overall_Max
0,2006-01-27 05:00:00+00:00,19.234521,19.743110,19.206267,19.778437,134520700,27,4,1,1,...,18.648235,19.630091,18.951475,18.951475,0.0,0.0,62.643770,97.775456,11.283761,448.369995
1,2006-01-30 05:00:00+00:00,19.651289,19.905584,19.623035,19.884378,103999200,30,0,1,1,...,18.718876,19.778437,19.078700,19.078700,0.0,0.0,65.659794,97.775456,11.283761,448.369995
2,2006-01-31 05:00:00+00:00,19.714849,20.046844,19.686595,19.806681,94841300,31,1,1,1,...,19.630091,19.884378,19.202651,19.202651,0.0,0.0,65.753562,97.775456,11.283761,448.369995
3,2006-02-01 05:00:00+00:00,19.750170,19.827871,19.608896,19.552387,68448800,1,2,1,2,...,19.778437,19.884378,19.295578,19.295578,0.0,0.0,60.807027,97.775456,11.283761,448.369995
4,2006-02-02 05:00:00+00:00,19.757235,19.771362,19.460558,19.453501,55073400,2,3,1,2,...,19.552387,19.884378,19.335087,19.335087,0.0,0.0,57.337032,97.775456,11.283761,448.369995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4623,2024-06-11 04:00:00+00:00,425.480011,432.820007,425.250000,441.059998,14551100,11,1,2,6,...,423.850006,432.679993,424.081913,424.081913,0.0,0.0,53.910602,97.775456,11.283761,448.369995
4624,2024-06-12 04:00:00+00:00,435.320007,443.399994,433.250000,441.579987,22366200,12,2,2,6,...,427.869995,441.059998,426.693926,426.693926,0.0,0.0,59.861540,97.775456,11.283761,448.369995
4625,2024-06-13 04:00:00+00:00,440.850006,443.390015,439.369995,442.570007,15960600,13,3,2,6,...,432.679993,441.579987,428.984089,428.984089,0.0,0.0,64.452810,97.775456,11.283761,448.369995
4626,2024-06-14 04:00:00+00:00,438.279999,443.140015,436.720001,448.369995,13582000,14,4,2,6,...,441.059998,442.570007,431.074231,431.074231,0.0,0.0,62.854782,97.775456,11.283761,448.369995


In [72]:
rfr_X = data.drop(["Close","Date"],axis = 1)
rfr_y = data["Close"]
X_train, X_test, y_train, y_test = train_test_split(rfr_X,rfr_y,test_size = 0.3,random_state = 101,shuffle = False)

In [73]:
rfr = RandomForestRegressor()
rfr.fit(X_train,y_train)
rfr

RandomForestRegressor()

In [74]:
forecast_preds = rfr.predict(X_test)
forecast_preds

array([101.20565483, 101.43849205, 104.28687042, ..., 107.09616013,
       106.95534798, 107.18345833])

In [75]:
print("Mean Absolute Error : ", mean_absolute_error(forecast_preds, y_test))
print("Root Mean Squared Error : ", np.sqrt(mean_squared_error(forecast_preds, y_test)))

Mean Absolute Error :  138.30573109499753
Root Mean Squared Error :  163.1977922263716


In [19]:
# plt.plot(data["Date"], data["Close"], label='Historical')

# plt.plot(data["Date"][-len(forecast_preds):], forecast_preds, label='Forecast')

# plt.legend()
# plt.show()

In [76]:
xgb = XGBRegressor(objective='reg:squarederror', n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)

# Boost the random forest regressor with XGBoost
boosted_rf = RandomForestRegressor(n_estimators=100, random_state=42)
boosted_rf.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [78]:
xgb_forecast_preds = xgb.predict(X_test)
forecast_preds
print("Mean Absolute Error : ", mean_absolute_error(forecast_preds, y_test))
print("Root Mean Squared Error : ", np.sqrt(mean_squared_error(forecast_preds, y_test)))

Mean Absolute Error :  138.30573109499753
Root Mean Squared Error :  163.1977922263716


In [79]:
forecast_preds

array([101.20565483, 101.43849205, 104.28687042, ..., 107.09616013,
       106.95534798, 107.18345833])